# 01 · Data preparation

Downloads a masked-face dataset, converts annotations to YOLO format, makes a stratified 70/15/15 split, writes `data.yaml`, previews augmentations and saves everything to Drive.

> **Runtime:** go to *Runtime → Change runtime type → T4 GPU* before running anything.
> Every cell below is safe to re-run; nothing is lost when Colab disconnects because all
> data, checkpoints and results live on your Google Drive.

## 0.1 GPU check
**What:** prints the GPU Colab assigned to this session.
**Why:** training on CPU takes hours instead of minutes; we warn loudly if no GPU is present.

In [ ]:
import subprocess, shutil

try:
    if shutil.which("nvidia-smi") is None:
        raise FileNotFoundError("nvidia-smi not found")
    print(subprocess.check_output(["nvidia-smi"], encoding="utf-8", errors="replace"))
    GPU_AVAILABLE = True
except Exception as exc:
    GPU_AVAILABLE = False
    print("=" * 70)
    print("WARNING: No GPU detected (", exc, ")")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    print("=" * 70)

## 0.2 Mount Google Drive & get the code
**What:** mounts Drive at `/content/drive`, then either uses a copy of the repo already on Drive
or clones it from GitHub into `/content`.
**Why:** Drive is the only storage that survives a runtime disconnect. Checkpoints, the prepared
dataset and evaluation outputs are all written under `SAVE_DIR`.

Edit `REPO_URL` once (your fork), or copy the repo folder to
`MyDrive/masked-face-detection/repo` and it will be picked up automatically.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Numbu-bit/masked-face-detection.git"   # <-- change if you fork
SAVE_DIR = "/content/drive/MyDrive/masked-face-detection"                  # everything persistent lives here
DRIVE_REPO = os.path.join(SAVE_DIR, "repo")
LOCAL_REPO = "/content/masked-face-detection"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab - using the current working directory as the repo.")
    SAVE_DIR = os.path.abspath("runs")

os.makedirs(SAVE_DIR, exist_ok=True)

if IN_COLAB:
    if os.path.isfile(os.path.join(DRIVE_REPO, "configs", "default.yaml")):
        REPO_DIR = DRIVE_REPO
        print("Using repo copy on Drive:", REPO_DIR)
    else:
        REPO_DIR = LOCAL_REPO
        if not os.path.isfile(os.path.join(REPO_DIR, "configs", "default.yaml")):
            if "YOUR_USERNAME" in REPO_URL:
                raise RuntimeError(
                    "Set REPO_URL to your GitHub fork above, OR copy the repository folder to "
                    f"{DRIVE_REPO} so the notebook can find configs/default.yaml.")
            rc = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
            if rc != 0:
                raise RuntimeError(f"git clone failed for {REPO_URL}. Is the repo public / URL correct?")
        else:
            os.system(f"git -C {REPO_DIR} pull -q")
else:
    REPO_DIR = os.getcwd() if os.path.isfile("configs/default.yaml") else os.path.abspath("..")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)
print("Persistent storage:", SAVE_DIR)

## 0.3 Install dependencies
**What:** installs the pinned versions from `requirements.txt` (quietly).
**Why:** pinning avoids the "it worked yesterday" class of Colab breakages.
Takes ~1–2 minutes on a fresh runtime; instant on re-runs.

In [ ]:
import subprocess, sys

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   check=True, capture_output=True, encoding="utf-8", errors="replace")
    print("Dependencies installed.")
except subprocess.CalledProcessError as exc:
    print("pip install failed. Last lines of the error:\n", exc.stderr[-2000:])
    raise RuntimeError("Fix the dependency error above, then re-run this cell.") from exc

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 0.4 Seeds & config
**What:** loads `configs/default.yaml` (the single source of truth for every hyper-parameter)
and seeds Python / NumPy / PyTorch / CUDA with `seed=42`.
**Why:** reproducible splits, reproducible training.

In [ ]:
import json
from src.utils import load_config, set_seed, get_device, resolve_save_dir

# Optional: MFD_OVERRIDES='{"epochs": 1}' in the environment (used by automated tests)
ENV_OVERRIDES = json.loads(os.environ.get("MFD_OVERRIDES", "{}"))
cfg = load_config(overrides={"save_dir": os.path.join(SAVE_DIR, "runs"), **ENV_OVERRIDES})
set_seed(cfg["seed"])
device = get_device()
RUN_DIR = resolve_save_dir(cfg) / cfg["run_name"]
DATA_ROOT = cfg["data_root"]
print("Model:", cfg["model_variant"], "| image size:", cfg["image_size"], "| batch:", cfg["batch_size"])
print("Run directory:", RUN_DIR)

## 1. Download the dataset — **choose ONE option**

| Option | Source | Credentials | Notes |
|---|---|---|---|
| **A** | Kaggle `andrewmvd/face-mask-detection` | **none needed** (public dataset); API token only as a fallback | 853 images, Pascal-VOC XML, 3 classes, ~4 K faces |
| **B** | Roboflow Universe "face mask detection" | API key | pre-split, YOLOv8 export, many projects to pick from |

Both cells detect missing credentials and tell you exactly what to do. Run only the one you want;
each sets `DATA_SOURCE` so the conversion step knows what layout to expect.

### Option A — Kaggle
**What:** downloads and unzips the dataset to `/content/data/raw`.
**Why no token first:** the dataset is public and the Kaggle CLI (≥ 2.x) allows anonymous
`datasets download`, so the cell simply tries that. Only if Kaggle refuses does it look for a token.

**If a token is needed** — Kaggle now shows the token as a *string* (`KGAT_…`) on
*kaggle.com → Settings → API → Generate New Token*; there is no longer a `kaggle.json` download.
The cell accepts it from any of these, in order:

1. Colab secret named `KAGGLE_API_TOKEN` (🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*) — recommended, never lands in Drive or the notebook
2. a text file `MyDrive/kaggle_token.txt` containing just the token
3. a legacy `MyDrive/kaggle.json` (`{"username": ..., "key": ...}`) if you still have one
4. a paste prompt (hidden input) — the value is then saved to `MyDrive/kaggle_token.txt` for next time

In [ ]:
import os, shutil, subprocess
from pathlib import Path

RAW_DIR = Path(DATA_ROOT) / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_SOURCE = None
DRIVE = Path("/content/drive/MyDrive")
KAGGLE_BIN = shutil.which("kaggle")   # the console script (NOT `python -m kaggle`, which disables anonymous mode)


def run_kaggle_download() -> subprocess.CompletedProcess:
    """Run `kaggle datasets download` with the current environment / ~/.kaggle contents."""
    return subprocess.run([KAGGLE_BIN, "datasets", "download", "-d", cfg["kaggle_dataset"],
                           "-p", str(RAW_DIR), "--unzip"],
                          capture_output=True, encoding="utf-8", errors="replace")


def find_kaggle_token() -> str | None:
    """Locate a Kaggle API token: Colab secret -> Drive text file -> legacy kaggle.json -> paste prompt."""
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    # 1. Colab secret
    if IN_COLAB:
        try:
            from google.colab import userdata
            tok = userdata.get("KAGGLE_API_TOKEN")
            if tok:
                return tok.strip()
        except Exception:
            pass
    # 2. token text file on Drive
    tok_file = DRIVE / "kaggle_token.txt"
    if tok_file.exists() and tok_file.read_text().strip():
        return tok_file.read_text().strip()
    # 3. legacy kaggle.json (username + key) -> just install it where the CLI looks
    for legacy in (DRIVE / "kaggle.json", Path("/content/kaggle.json")):
        if legacy.exists():
            shutil.copy(legacy, kaggle_dir / "kaggle.json")
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
            return ""  # empty string = "legacy file installed, no env token"
    # 4. ask
    if IN_COLAB:
        from getpass import getpass
        print("=" * 70)
        print("Kaggle needs authentication. Get a token at kaggle.com -> Settings -> API ->")
        print("'Generate New Token', copy the KGAT_... string and paste it below (input is hidden).")
        print("It will be saved to MyDrive/kaggle_token.txt so you never need to do this again.")
        print("=" * 70)
        tok = getpass("KAGGLE_API_TOKEN: ").strip()
        if tok:
            tok_file.write_text(tok)
            return tok
    return None


if (RAW_DIR / "annotations").exists() and (RAW_DIR / "images").exists():
    print("Dataset already present at", RAW_DIR, "- skipping download.")
else:
    if KAGGLE_BIN is None:
        raise RuntimeError("The `kaggle` CLI is not on PATH. Re-run the dependency cell (0.3).")
    print("Downloading", cfg["kaggle_dataset"], "(anonymous, public dataset) ...")
    proc = run_kaggle_download()
    if proc.returncode != 0:
        print("Anonymous download refused - looking for a Kaggle API token ...")
        token = find_kaggle_token()
        if token is None:
            raise RuntimeError(
                "No Kaggle token available. Add a Colab secret KAGGLE_API_TOKEN (or MyDrive/kaggle_token.txt), "
                "or use Option B (Roboflow).\nKaggle said:\n" + (proc.stderr or proc.stdout)[-1500:])
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token          # new-style token
        proc = run_kaggle_download()
        if proc.returncode != 0:
            raise RuntimeError(
                "Kaggle download failed even with a token:\n" + (proc.stderr or proc.stdout)[-1500:] +
                "\nCommon fixes: regenerate the token at kaggle.com/settings/api, make sure the dataset "
                "page loads while logged in, or use Option B (Roboflow).")
    print(proc.stdout[-500:])

DATA_SOURCE = "kaggle_voc"
n_img = len(list((RAW_DIR / "images").glob("*")))
n_xml = len(list((RAW_DIR / "annotations").glob("*.xml")))
if n_img == 0 or n_xml == 0:
    raise RuntimeError(f"Download finished but {RAW_DIR} has {n_img} images / {n_xml} annotations. "
                       "Delete the folder and re-run this cell.")
print(f"{n_img} images, {n_xml} annotation files in {RAW_DIR}")

### Option B — Roboflow (alternative)
**What:** downloads a YOLOv8-format export from Roboflow Universe.
**Why:** Roboflow projects come pre-labelled in YOLO format and are often larger than the Kaggle set.
Uncomment, paste your API key (*Roboflow → Settings → API Keys*) and adjust `WORKSPACE / PROJECT / VERSION`
to the project you picked on https://universe.roboflow.com (search "face mask detection").

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)  # installed on demand
# from roboflow import Roboflow
# from pathlib import Path
#
# ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"
# WORKSPACE, PROJECT, VERSION = "workspace-slug", "face-mask-detection", 1
#
# RAW_DIR = Path(DATA_ROOT) / "raw_roboflow"
# if "PASTE" in ROBOFLOW_API_KEY:
#     raise RuntimeError("Paste your Roboflow API key into ROBOFLOW_API_KEY above.")
# try:
#     rf = Roboflow(api_key=ROBOFLOW_API_KEY)
#     dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location=str(RAW_DIR))
# except Exception as exc:
#     raise RuntimeError(f"Roboflow download failed: {exc}. Check the key and workspace/project/version.") from exc
# DATA_SOURCE = "roboflow_yolo"
# print("Downloaded to", RAW_DIR)

## 2. Explore the raw data
**What:** class histogram, image-size distribution and a few ground-truth examples.
**Why:** you want to know *before* training that the third class (`mask_worn_incorrectly`) is rare —
it is in every public mask dataset, and it explains the per-class F1 you will see later.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from collections import Counter
from src.dataset import (collect_voc_samples, collect_yolo_samples, read_roboflow_names,
                         build_class_remap, yolo_to_xyxy)
from src.utils import draw_detections, bgr_to_rgb

if DATA_SOURCE == "kaggle_voc":
    samples = collect_voc_samples(RAW_DIR, cfg)
elif DATA_SOURCE == "roboflow_yolo":
    remap = build_class_remap(read_roboflow_names(RAW_DIR / "data.yaml"), cfg["class_names"])
    print("class remap (roboflow -> ours):", remap)
    samples = collect_yolo_samples(RAW_DIR, remap=remap)
else:
    raise RuntimeError("Run Option A or Option B first so DATA_SOURCE is set.")

counts = Counter(lbl[0] for s in samples for lbl in s.labels)
df = pd.DataFrame({"class": cfg["class_names"], "boxes": [counts.get(i, 0) for i in range(cfg["num_classes"])]})
display(df)
print(f"{len(samples)} images, {sum(counts.values())} boxes, "
      f"{sum(1 for s in samples if not s.labels)} images without boxes")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(df["class"], df["boxes"], color=["green", "red", "orange"]); axes[0].set_title("Boxes per class")
sizes = [cv2.imread(str(s.image_path)).shape[:2] for s in random.Random(cfg["seed"]).sample(samples, min(100, len(samples)))]
axes[1].scatter([w for h, w in sizes], [h for h, w in sizes], s=8); axes[1].set_title("Image sizes (100 random)")
axes[1].set_xlabel("width"); axes[1].set_ylabel("height"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, s in zip(axes.ravel(), random.Random(cfg["seed"]).sample(samples, 8)):
    img = cv2.imread(str(s.image_path)); boxes, ids = yolo_to_xyxy(s.labels, img.shape[1], img.shape[0])
    ax.imshow(bgr_to_rgb(draw_detections(img, boxes, ids, [1.0] * len(ids), cfg, show_summary=False)))
    ax.set_title(s.image_path.name, fontsize=8); ax.axis("off")
plt.suptitle("Ground truth (green=with_mask, red=without_mask, orange=incorrect)"); plt.tight_layout(); plt.show()

## 3. Convert → stratified split → YOLO layout → `data.yaml`
**What:** writes `/content/data/{train,val,test}/{images,labels}` (70/15/15, stratified by each
image's dominant class), resizes images so the long side is ≤ `image_size`, and writes `data.yaml`.
**Why:** this is exactly the layout Ultralytics expects; resizing up-front shrinks the Drive
backup and speeds up data loading.

In [ ]:
from pathlib import Path
from src.dataset import stratified_split, write_split, write_data_yaml, validate_yolo_dataset, dataset_statistics

split = stratified_split(samples, cfg["split_ratios"], cfg["seed"])
data_root = write_split(split, Path(DATA_ROOT), image_size=cfg["image_size"])
DATA_YAML = write_data_yaml(data_root, cfg["class_names"])

problems = validate_yolo_dataset(data_root, cfg["num_classes"])
if problems:
    print(f"{len(problems)} problems found (showing 10):"); print("\n".join(problems[:10]))
    raise RuntimeError("Dataset validation failed - see above.")
print("Dataset validation passed.")
display(pd.DataFrame(dataset_statistics(data_root, cfg["class_names"])).T)
print(open(DATA_YAML).read())

## 4. Preview the Albumentations pipeline
**What:** applies the training augmentations (brightness/contrast, flip, scale, blur, CLAHE,
colour-jitter, coarse dropout) to one image several times, with boxes.
**Why:** a visual check that boxes follow the image under every transform — the most common
silent bug in detection pipelines.

Note: Ultralytics applies its own strong augmentation (mosaic, mixup, HSV) during YOLO training
using the same probabilities from the config; this Albumentations pipeline is used by the
`MaskDataset` class (SSD fallback) and here for inspection.

In [ ]:
import numpy as np
from src.dataset import MaskDataset
from src.utils import draw_detections, bgr_to_rgb, rgb_to_bgr

ds = MaskDataset(data_root, "train", cfg, augment=True)
idx = random.Random(cfg["seed"]).randrange(len(ds))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.ravel():
    tensor, target = ds[idx]
    img = (tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    boxes = target["boxes"].numpy().tolist(); ids = (target["labels"].numpy() - 1).tolist()
    ax.imshow(bgr_to_rgb(draw_detections(rgb_to_bgr(img), boxes, ids, [1.0] * len(ids), cfg, show_summary=False)))
    ax.axis("off")
plt.suptitle(f"8 augmented views of {ds.images[idx].name}"); plt.tight_layout(); plt.show()

## 5. Back up the prepared dataset to Drive
**What:** zips `/content/data/{train,val,test,data.yaml}` to `MyDrive/masked-face-detection/data_prepared.zip`.
**Why:** notebooks 02–04 restore from this zip automatically, so you never re-download or re-split
after a disconnect.

In [ ]:
import zipfile
from tqdm.auto import tqdm

DATA_ZIP = Path(SAVE_DIR) / "data_prepared.zip"
files = [p for split_name in ("train", "val", "test") for p in (data_root / split_name).rglob("*") if p.is_file()]
files.append(DATA_YAML)
try:
    with zipfile.ZipFile(DATA_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in tqdm(files, desc="zipping"):
            zf.write(p, arcname=str(Path("data") / p.relative_to(data_root)))
    print(f"Saved {DATA_ZIP} ({DATA_ZIP.stat().st_size / 1e6:.1f} MB)")
except OSError as exc:
    raise RuntimeError(f"Could not write to Drive: {exc}. Is Drive mounted and not full?") from exc

from src.utils import free_memory
free_memory()
print("Done - continue with 02_model_training.ipynb")